In [1]:
!pip install -q torchmetrics piq

In [2]:
import os
from google.colab import drive
from datasets import load_from_disk
from dotenv import load_dotenv
import numpy as np
from PIL import Image
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torchmetrics
import piq
from typing import Dict




In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
TRAIN_DENOISER=False
DATASET_DIR = "/content/drive/MyDrive/AAI-521 Final Project/denoiser" if (TRAIN_DENOISER) else "/content/drive/MyDrive/AAI-521 Final Project/super_res"
os.makedirs(DATASET_DIR, exist_ok=True)


In [5]:
env_path = "/content/drive/MyDrive/AAI-521 Final Project/hf_login.env"
# Open env file
load_dotenv(env_path)
# Access token from environment
hf_key = os.getenv("HF_TOKEN")
print("HF key loaded:", hf_key is not None)

HF key loaded: True


In [6]:
from datasets import load_dataset

#train = load_dataset("detection-datasets/coco", split="train[:200]")
#val   = load_dataset("detection-datasets/coco", split="train[200:250]")
#test  = load_dataset("detection-datasets/coco", split="train[250:300]")

# the dataset is in denoiser dir, future: move to coco_dataset
def load_or_create_coco_dataset():
    if os.path.exists(DATASET_DIR):
        print(f"📂 Loading dataset from disk: {DATASET_DIR}")
        return load_from_disk("/content/drive/MyDrive/AAI-521 Final Project/denoiser")
    ds = load_dataset("detection-datasets/coco", split="train[:3000]")
    ds.save_to_disk(DATASET_DIR)

ds = load_or_create_coco_dataset()

#ds = load_dataset("detection-datasets/coco", split="train[:3000]")

splits = ds.train_test_split(test_size=0.2, seed=42)
train = splits["train"]
temp = splits["test"]

# Split temp into val and test (50/50)
val_test = temp.train_test_split(test_size=0.5, seed=42)
val = val_test["train"]
test = val_test["test"]

📂 Loading dataset from disk: /content/drive/MyDrive/AAI-521 Final Project/super_res


In [7]:
train

Dataset({
    features: ['image_id', 'image', 'width', 'height', 'objects'],
    num_rows: 2400
})

In [8]:
test

Dataset({
    features: ['image_id', 'image', 'width', 'height', 'objects'],
    num_rows: 300
})

In [9]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms as T
import random
from PIL import Image


class CocoDenoiseHF(Dataset):
    def __init__(self, hf_dataset, resolution=256, noise_std=0.3):
        self.ds = hf_dataset
        self.noise_std = noise_std

        self.transform = T.Compose([
            T.Resize((resolution, resolution)),
            T.ToTensor(),
            T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Scale to [-1, 1] for Stable Diffusion
        ])

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]

        # Use the image directly
        img = item["image"]

        # Convert to RGB if grayscale
        if img.mode != 'RGB':
            img = img.convert('RGB')

        img = self.transform(img)

        # Create noisy version (add noise in [-1, 1] space)
        noise = torch.randn_like(img) * self.noise_std
        noisy = (img + noise).clamp(-1, 1)  # Clamp to [-1, 1] since we normalized

        return {
            "pixel_values": noisy,  # What the training script expects
            "clean_image": img      # Optional: keep for reference
        }

In [10]:
# For super resolution
class CocoSuperResHF(Dataset):
    """
    COCO dataset for Super-Resolution fine-tuning.
    Produces:
        - 'lr_image'  : upsampled low-res (model input)
        - 'hr_image'  : high-res ground truth (target)
    """

    def __init__(self, hf_dataset, hr_resolution=256, scale_factor=4):
        self.ds = hf_dataset
        self.scale = scale_factor
        self.hr_resolution = hr_resolution

        self.hr_transform = T.Compose([
            T.Resize((hr_resolution, hr_resolution), interpolation=Image.BICUBIC),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

        # low-res is hr_resolution / scale, then upsampled back to hr_resolution
        self.lr_down = T.Resize((hr_resolution // scale_factor,
                                 hr_resolution // scale_factor),
                                 interpolation=Image.BICUBIC)

        self.lr_up = T.Resize((hr_resolution, hr_resolution),
                              interpolation=Image.BICUBIC)

        self.to_norm = T.Compose([
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        img = item["image"]

        if img.mode != "RGB":
            img = img.convert("RGB")

        # High-res clean image
        hr = self.hr_transform(img)

        # Create low-res → upsampled image
        lr_small = self.lr_down(img)
        lr_upsampled = self.lr_up(lr_small)

        # Normalize LR to match UNet input format [-1,1]
        lr = self.to_norm(lr_upsampled)

        return {
            "pixel_values": lr,      # Model input
            "clean_image": hr        # Target latent for noise prediction loss
        }

In [11]:
import torch
import gc

# Clear all GPU memory
torch.cuda.empty_cache()
gc.collect()


126

In [ ]:
# fine tune specific application denoise or super resolution.py
import torch
import torchvision.transforms as T
from torch import nn
from torch.utils.data import DataLoader
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from transformers import CLIPTokenizer, CLIPTextModel
from accelerate import Accelerator
import random
import matplotlib.pyplot as plt

loss_history = []

def smooth_curve(values, alpha=0.9):
    smoothed = []
    last = values[0]
    for v in values:
        last = alpha * last + (1 - alpha) * v
        smoothed.append(last)
    return smoothed

def main():
    accelerator = Accelerator(mixed_precision="fp16")

    # ----- Load pretrained SD components -----
    model_name = "runwayml/stable-diffusion-v1-5"

    vae = AutoencoderKL.from_pretrained(model_name, subfolder="vae")
    text_encoder = CLIPTextModel.from_pretrained(model_name, subfolder="text_encoder")
    tokenizer = CLIPTokenizer.from_pretrained(model_name, subfolder="tokenizer")
    unet = UNet2DConditionModel.from_pretrained(model_name, subfolder="unet")
    noise_scheduler = DDPMScheduler.from_pretrained(model_name, subfolder="scheduler")

    # After loading UNet
    unet.enable_gradient_checkpointing()
    #unet.enable_xformers_memory_efficient_attention()
    # Freeze VAE + text encoder (only train UNet)
    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)

    # ----- Dataset -----
    ds = CocoDenoiseHF(train) if (TRAIN_DENOISER) else CocoSuperResHF(train)
    dl = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

    # Optimizer
    optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-5)

    unet, optimizer, dl, vae, text_encoder = accelerator.prepare(
        unet, optimizer, dl, vae, text_encoder
    )

    #start_epoch = 0
    #if args.resume_from:
    #accelerator.print(f"🔄 Resuming from checkpoint")
    #accelerator.load_state(f"checkpoint-epoch-{start_epoch}")

    # Training
    num_epochs = 5
    vae_scale = 0.18215

    for epoch in range(num_epochs):
        for step, batch in enumerate(dl):
            # DEBUG: Print shapes on first iteration
            if step == 0:
                accelerator.print(f"\n=== DEBUG: First batch shapes ===")
                accelerator.print(f"Input pixel_values shape: {batch['pixel_values'].shape}")


            # for super resolution use clean_image instead of pixel_value
            # 1 — image → latent
            with torch.no_grad():
                batch_name = "pixel_values" if (TRAIN_DENOISER) else "clean_image"
                latents = vae.encode(batch[batch_name]).latent_dist.sample()
                latents *= vae_scale

            # DEBUG: Print latent shape on first iteration
            if step == 0:
                accelerator.print(f"Latents shape: {latents.shape}")

            # 2 — sample random diffusion timestep
            bsz = latents.shape[0]
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                      (bsz,), device=latents.device).long()

            # 3 — add noise
            noise = torch.randn_like(latents)
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # 4 — text conditioning
            tokens = tokenizer(
                [""] * bsz,  # Fixed: needs to match batch size
                padding="max_length",
                max_length=tokenizer.model_max_length,
                truncation=True,
                return_tensors="pt"
            ).input_ids.to(latents.device)

            with torch.no_grad():
                text_embeds = text_encoder(tokens).last_hidden_state

            # DEBUG: Print text embeds shape on first iteration
            if step == 0:
                accelerator.print(f"Text embeds shape: {text_embeds.shape}")
                accelerator.print(f"Noisy latents shape: {noisy_latents.shape}")
                accelerator.print(f"Timesteps shape: {timesteps.shape}")
                accelerator.print(f"=== End DEBUG ===\n")

            # 5 — predict noise
            noise_pred = unet(
                noisy_latents, timesteps, encoder_hidden_states=text_embeds
            ).sample

            # 6 — loss = MSE(predicted_noise, true_noise)
            loss = nn.functional.mse_loss(noise_pred, noise)

            if accelerator.is_main_process:
              loss_history.append(loss.item())

            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

            if step % 50 == 0:
                accelerator.print(f"Epoch {epoch} | Step {step} | Loss {loss.item():.4f}")

        accelerator.wait_for_everyone()
        accelerator.save_state(f"checkpoint-epoch-{TRAIN_DENOISER}-{epoch}")

        # Plot loss curve
        if accelerator.is_main_process:
          plt.figure(figsize=(10, 5))
          plt.plot(loss_history, label="Raw Loss", alpha=0.3)
          plt.plot(smooth_curve(loss_history), label="Smoothed Loss (EMA)")
          plt.title("Training Loss Curve")
          plt.xlabel("Step")
          plt.ylabel("Loss")
          plt.grid(True)
          plt.legend()
          plt.savefig(f"loss_curve_smooth_{TRAIN_DENOISER}.png")
          plt.close()

          accelerator.print("Saved loss curve to loss_curve.png")


        if accelerator.is_main_process:

          accelerator.print("Saving fine-tuned model...")

          # Create output folder
          output_dir = DATASET_DIR + "/sd_finetuned_denoiser" if (TRAIN_DENOISER) else DATASET_DIR + "/sd_finetuned_super_res"
          os.makedirs(output_dir, exist_ok=True)

          # Save UNet weights
          unet.save_pretrained(f"{output_dir}/unet")

          # Save scheduler too (optional but recommended)
          noise_scheduler.save_pretrained(f"{output_dir}/scheduler")

          accelerator.print(f"Model saved to: {output_dir}")

          # How to load
          #from diffusers import UNet2DConditionModel, DDPMScheduler
          #unet = UNet2DConditionModel.from_pretrained("sd_finetuned_denoiser/unet")
          #scheduler = DDPMScheduler.from_pretrained("sd_finetuned_denoiser/scheduler")



if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



=== DEBUG: First batch shapes ===
Input pixel_values shape: torch.Size([1, 3, 256, 256])
Latents shape: torch.Size([1, 4, 32, 32])
Text embeds shape: torch.Size([1, 77, 768])
Noisy latents shape: torch.Size([1, 4, 32, 32])
Timesteps shape: torch.Size([1])
=== End DEBUG ===

Epoch 0 | Step 0 | Loss 0.2406
Epoch 0 | Step 50 | Loss 0.0034
Epoch 0 | Step 100 | Loss 0.1023
Epoch 0 | Step 150 | Loss 0.0797
Epoch 0 | Step 200 | Loss 0.1394
Epoch 0 | Step 250 | Loss 0.0699
Epoch 0 | Step 300 | Loss 0.1904
Epoch 0 | Step 350 | Loss 0.0551
Epoch 0 | Step 400 | Loss 0.0044
Epoch 0 | Step 450 | Loss 0.7162
Epoch 0 | Step 500 | Loss 0.3556
Epoch 0 | Step 550 | Loss 0.4508
Epoch 0 | Step 600 | Loss 0.0595
Epoch 0 | Step 650 | Loss 0.2806
Epoch 0 | Step 700 | Loss 0.0219
Epoch 0 | Step 750 | Loss 0.1218
Epoch 0 | Step 800 | Loss 0.2136
Epoch 0 | Step 850 | Loss 0.0444
Epoch 0 | Step 900 | Loss 0.0122
Epoch 0 | Step 950 | Loss 0.2101
Epoch 0 | Step 1000 | Loss 0.1801
Epoch 0 | Step 1050 | Loss 0.0223

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(loss_history, label="Raw Loss", alpha=0.3)
plt.plot(smooth_curve(loss_history), label="Smoothed Loss (EMA)")
plt.title("Training Loss Curve")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.savefig(f"loss_curve_smooth_{TRAIN_DENOISER}.png")
plt.close()

In [ ]:
# For running metrics this is the diffuser pipeline.
denoiser_model_id = "runwayml/stable-diffusion-v1-5"
from diffusers import StableDiffusionImg2ImgPipeline, UNet2DConditionModel, AutoencoderKL, DDPMScheduler
import torch

print(f"Loading diffusion denoising pipeline: {denoiser_model_id}")

denoise_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    denoiser_model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    safety_checker=None   # often needed for img2img in local environments
)

# Custom fine tuned model
#DATASET_DIR = "/content/drive/MyDrive/AAI-521 Final Project/denoiser/sd_finetuned_denoiser/unet"
#print("Loading fine-tuned UNet...")
#fine_tuned_unet = UNet2DConditionModel.from_pretrained(
#    DATASET_DIR,
#    torch_dtype=denoise_pipe.unet.dtype,
#)

#denoise_pipe.unet = fine_tuned_unet
print("Fine-tuned UNet loaded.")

denoise_pipe = denoise_pipe.to("cuda" if torch.cuda.is_available() else "cpu")

def diffusion_denoise(
    pil_image,
    prompt="",
    negative_prompt="grain, noise, artifacts, blur",
    denoise_strength=0.25,        # Lower = closer to original; higher = stronger denoising
    guidance_scale=7.0,
    num_inference_steps=25
):
    """
    Perform diffusion-based denoising using Stable Diffusion img2img.
    The image is slightly "nudged" toward a clean version of itself.
    """
    print("Running diffusion denoising...")

    # Ensure correct size for the diffusion pipeline
    image_resized = pil_image.resize(IMG_SIZE)

    result = denoise_pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image_resized,
        strength=denoise_strength,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    )

    denoised_image = result.images[0]

    print("Denoising complete.")
    return denoised_image


In [ ]:
# --- Helper functions (ensuring all dependencies are met in this block) ---

def _device():
    return "cuda" if torch.cuda.is_available() else "cpu"

def pil_to_tensor(img: Image.Image) -> torch.Tensor:
    """Convert PIL → C×H×W float tensor in [0, 1] on the evaluation device."""
    arr = np.array(img).astype(np.float32) / 255.0          # H×W×C, 0–1
    tensor = torch.from_numpy(arr).permute(2, 0, 1)        # C×H×W
    return tensor.to(_device())

def compute_metrics(
    clean: torch.Tensor,
    denoised: torch.Tensor,
) -> Dict[str, float]:
    """
    Compute a dictionary of metrics for a *single* image pair.
    All tensors must be on the same device and have shape (C, H, W) in [0,1].
    """
    # Ensure the tensors are 4–D (batch dim) for torchmetrics
    clean = clean.unsqueeze(0)
    denoised = denoised.unsqueeze(0)

    # PSNR (higher = better) - Updated API
    from torchmetrics.image import PeakSignalNoiseRatio
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(_device())
    psnr = psnr_metric(denoised, clean).item()

    # SSIM (higher = better) - Updated API
    from torchmetrics.image import StructuralSimilarityIndexMeasure
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(_device())
    ssim = ssim_metric(denoised, clean).item()

    # LPIPS (lower = better) – Updated API for piq
    from piq import LPIPS
    lpips_metric = LPIPS(reduction='mean').to(_device())
    # piq LPIPS expects inputs in [0, 1] range
    lpips = lpips_metric(denoised, clean).item()

    return {"psnr": psnr, "ssim": ssim, "lpips": lpips}

def tensor_to_pil(tensor_img):
    """Converts a C x H x W tensor in [-1, 1] to a PIL Image."""
    # Denormalize from [-1, 1] to [0, 1]
    tensor_img = (tensor_img / 2 + 0.5).clamp(0, 1)
    # Convert to H x W x C, then to numpy, then to PIL
    np_img = tensor_img.permute(1, 2, 0).cpu().numpy()
    np_img = (np_img * 255).astype(np.uint8)
    return Image.fromarray(np_img)


# --- Evaluation Setup ---

IMG_SIZE = (256, 256) # Based on model resolution set during training

# Alias diffusion_denoise from previous cell
denoise_image = diffusion_denoise

# Denoise keyword arguments
denoise_kwargs = {
    "prompt": "", # Empty prompt for pure denoising
    "negative_prompt": "grain, noise, artifacts, blur, bad quality, ugly, watermark",
    "denoise_strength": 0.3, # This can be tuned
    "guidance_scale": 7.5,
    "num_inference_steps": 25
}

# 1. Create dataset and dataloader for the test split
test_dataset = CocoDenoiseHF(test, resolution=IMG_SIZE[0])
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2) # Batch size 1 for evaluation

results = []

print("Starting evaluation on the test set...")
for i, batch in enumerate(tqdm(test_dataloader, desc="Evaluating Test Set")):
    # Move tensors to the correct device
    noisy_tensor = batch["pixel_values"].to(_device()) # This is the input to the denoiser
    clean_tensor_target = batch["clean_image"].to(_device()) # This is the ground truth clean image

    # Convert noisy tensor to PIL for diffusion_denoise (assuming batch size is 1 for evaluation)
    noisy_pil = tensor_to_pil(noisy_tensor[0])

    # -------------------------------------------------
    # 2‼️ Denoise (re‑uses the shared pipeline)
    # -------------------------------------------------
    denoised_pil = denoise_image(noisy_pil, **denoise_kwargs)

    # -------------------------------------------------
    # 3‼️ Convert to torch tensors for metrics
    # -------------------------------------------------
    # Convert clean_tensor_target from [-1, 1] to [0, 1] for compute_metrics
    clean_tensor_for_metrics = (clean_tensor_target[0] / 2 + 0.5).clamp(0, 1)

    # Convert denoised PIL image to tensor in [0, 1]
    denoised_tensor_for_metrics = pil_to_tensor(denoised_pil)

    # -------------------------------------------------
    # 4‼️ Compute metrics
    # -------------------------------------------------
    metric_dict = compute_metrics(clean_tensor_for_metrics, denoised_tensor_for_metrics)
    metric_dict["index"] = i # Use index as identifier if no filename is available
    results.append(metric_dict)

print("\nEvaluation complete. Aggregating results...")

# Aggregate and print results
if results:
    avg_psnr = sum(r["psnr"] for r in results) / len(results)
    avg_ssim = sum(r["ssim"] for r in results) / len(results)
    avg_lpips = sum(r["lpips"] for r in results) / len(results)
    print(f"Average PSNR: {avg_psnr:.4f}")
    print(f"Average SSIM: {avg_ssim:.4f}")
    print(f"Average LPIPS: {avg_lpips:.4f}")
else:
    print("No results to aggregate.")


#Evaluation complete. Fine tuned results...
# Test set is 300 images
#Average PSNR: 15.7459 (higher is better)
#Average SSIM: 0.1540. (higher is better)
#Average LPIPS: 0.6604 (lower is better)

# Now for the not fine tuned
#Evaluation complete. Aggregating results...
#Average PSNR: 12.7420 (higher is better)
#Average SSIM: 0.1270. (higher is better)
#Average LPIPS: 0.7009 (lower is better)
